In [1]:
import pandas as pd
import numpy as np
from zipfile import ZipFile
from pathlib import Path
import re
from io import BytesIO, TextIOWrapper
from tqdm import tqdm
from collections import defaultdict
from pathlib import Path
import glob
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.interpolate import griddata
from scipy import sparse
import math
import json
import time
import h5py
from pyproj import CRS, Transformer
import rasterio
import rasterio.fill
from timezonefinder import TimezoneFinder
import pytz
from datetime import datetime, timedelta
import xarray as xr
import cdsapi
import zipfile
from dateutil.relativedelta import relativedelta
import dask
import dask.array
import xdem

# Loading files

In [2]:
subset_feature_list = ['ID', 'lon', 'lat', 'fireday', 'year', 'DOB', 'firearea', 'cumuarea', 'prec', 'tmax', 'ws', 'rh', 
                           'dem', 'slope', 'aspect', 
                           'Biomass', 'Closure', 'prcB', 'prcC']

In [3]:
fire_growth_2024 = pd.read_csv('Fire growth points/Firegrowth_pts_v1_1_2024/Firegrowth_pts_v1_1_2024.csv', usecols=subset_feature_list)

In [4]:
def lonlat_to_canada_lambert(df, lon_col='lon', lat_col='lat'):

    """
    Adds the projected coordinates in meters
    
    :param df: Dataframe
    :param lon_col: The longitude column name
    :param lat_col: The latitude column name
    """

    if lon_col not in df.columns or lat_col not in df.columns:
        raise ValueError(f"DataFrame must contain columns '{lon_col}' and '{lat_col}'")

    source_crs = CRS.from_epsg(4269)   # NAD83 geographic (degrees)
    target_crs = CRS.from_epsg(3347)   # NAD83 / Canada Lambert (meters)

    transformer = Transformer.from_crs(source_crs, target_crs, always_xy=True)

    lons = df[lon_col].to_numpy(dtype=float)
    lats = df[lat_col].to_numpy(dtype=float)

    # Transform to meters coordinates
    eastings, northings = transformer.transform(lons, lats)

    df['easting'] = eastings
    df['northing'] = northings
    df.attrs['target_crs'] = target_crs.to_string()

    return df, target_crs

In [5]:
fire_growth_2024, used_crs = lonlat_to_canada_lambert(fire_growth_2024)

In [6]:
fire_growth_2024.head()

,ID,DOB,year,fireday,firearea,prec,tmax,ws,rh,Biomass,...,prcB,prcC,dem,slope,aspect,cumuarea,lon,lat,easting,northing
0,2024_1,225,2024,1,311.04,0.0018,32.062402,13.14911,35.655201,86.222221,...,13.000000,87.000000,303.444458,2.517298,34.278625,311.04,-116.106160,60.100154,4.920918e+06,2.890838e+06
1,2024_1,225,2024,1,311.04,0.0018,32.062402,13.14911,35.655201,89.888885,...,15.666667,84.333333,301.222229,2.387177,34.278625,311.04,-116.104586,60.100425,4.921008e+06,2.890834e+06
2,2024_1,225,2024,1,311.04,0.0018,32.062402,13.14911,35.655201,93.000000,...,46.777779,53.222221,302.000000,1.103502,74.148155,311.04,-116.110339,60.098556,4.920644e+06,2.890762e+06
3,2024_1,225,2024,1,311.04,0.0018,32.062402,13.14911,35.655201,99.666664,...,47.111111,52.888889,302.111115,1.740722,5.107979,311.04,-116.108765,60.098826,4.920734e+06,2.890757e+06
4,2024_1,225,2024,1,311.04,0.0018,32.062402,13.14911,35.655201,85.222221,...,24.000000,76.000000,306.666656,1.065100,5.107979,311.04,-116.107191,60.099097,4.920824e+06,2.890753e+06


# Fire Subsets

In [7]:
FIRE_SUBSETS_FOLDER = 'Fire_Subsets'
if not os.path.exists(FIRE_SUBSETS_FOLDER):
    os.makedirs(FIRE_SUBSETS_FOLDER)
    print(f"Created directory: {FIRE_SUBSETS_FOLDER}")

In [8]:
def save_fire_subsets(df, fire_ids, output_folder):
    
    """
    Save the target fires as separate csvs
    
    :param df: Dataframe
    :param fire_ids: List of fire IDs
    :param output_folder: Folder of fires' csvs
    """

    for fire_id in fire_ids:
        # Filter the dataframe for the specific ID
        subset = df[df['ID'] == fire_id]
        
        if not subset.empty:
            # Define the filename
            file_name = f"subset_fire_{fire_id}.csv"
            file_path = os.path.join(output_folder, file_name)
            
            # Save to CSV
            subset.to_csv(file_path, index=False)
            print(f"Saved: {file_path}")
        else:
            print(f"Warning: Fire ID {fire_id} not found in the dataframe.")

In [9]:
# Quick verification
subsets_ids = ['2024_188']
save_fire_subsets(fire_growth_2024, subsets_ids, FIRE_SUBSETS_FOLDER)

Saved: Fire_Subsets\subset_fire_2024_188.csv


# Visualization functions

In [10]:
def plot_original_scatter(df, fire_id, lon_col="lon", lat_col="lat", area_col="firearea"):
    """
    Generate a scatter plot of a specific fire
    
    :param df: Dataframe
    :param fire_id: Fire ID
    :param lon_col: The longitude column name
    :param lat_col: The latitude column name
    :param area_col: Fire area feature (firearea or cumuarea)
    """

    # Filter rows for the chosen fire
    df_fire = df[df["ID"] == fire_id]

    if df_fire.empty:
        print(f"No data found for fire_ID = {fire_id}")
        return

    plt.figure(figsize=(10,10))
    
    # Scatter plot
    plt.scatter(
        df_fire[lon_col],
        df_fire[lat_col],
        s=30,
        marker='s',
        c=df_fire[area_col],
        cmap="viridis"
    )

    plt.colorbar(label="Fire Area")
    plt.xlabel(lon_col)
    plt.ylabel(lat_col)
    plt.title(f"Fire Area Scatter for Fire ID {fire_id}")
    plt.show()

In [11]:
def plot_original_scatter_by_day(df, fire_id, fireday,
                                 lon_col="lon", lat_col="lat", area_col="firearea"):
    """
    Generate a scatter plot of a fire's specific fireday
    
    :param df: Dataframe
    :param fire_id: Fire ID
    :param fireday: Fire day
    :param lon_col: The longitude column name
    :param lat_col: The latitude column name
    :param area_col: Fire area feature (firearea or cumuarea)
    """

    # Filter all rows for the fire (all days)
    df_fire_full = df[df["ID"] == fire_id]

    if df_fire_full.empty:
        print(f"No data found for fire_ID = {fire_id}")
        return

    # Compute global bounds for the entire fire
    min_lon, max_lon = df_fire_full[lon_col].min(), df_fire_full[lon_col].max()
    min_lat, max_lat = df_fire_full[lat_col].min(), df_fire_full[lat_col].max()

    # Filter the selected day
    df_fire_day = df_fire_full[df_fire_full["fireday"] == fireday]

    if df_fire_day.empty:
        print(f"No data found for fire_ID = {fire_id} on fireday {fireday}")
        return

    plt.figure(figsize=(10,10))

    plt.scatter(
        df_fire_day[lon_col],
        df_fire_day[lat_col],
        s=30,
        marker='s',
        c=df_fire_day[area_col],
        cmap="viridis"
    )

    # Fix the axis limits to full fire extent
    plt.xlim(min_lon, max_lon)
    plt.ylim(min_lat, max_lat)

    plt.colorbar(label="Fire Area")
    plt.xlabel(lon_col)
    plt.ylabel(lat_col)
    plt.title(f"Fire Area Scatter for Fire ID {fire_id}, Day {fireday}")
    plt.show()

In [12]:
def visualize_grid_values_by_image(grid, feature, fire_ID, fireday, patch):

    """
    Generates a heatmap visualization of a specific fire patch (grid)
    
    :param grid: feature's 2D array
    :param grid: feature to plot
    :param fire_ID: Fire ID
    :param fireday: Fire day
    :param patch: patch index relative to the entire fire
    """

    grid = np.asarray(grid)
    
    plt.figure(figsize=(7, 7))

    im = plt.imshow(grid, cmap='hot', origin='lower', interpolation='none')
    
    plt.title(f"{feature} - Fire {fire_ID} day {fireday} patch {patch}")
    plt.xlabel("x index")
    plt.ylabel("y index")
    
    plt.colorbar(im, fraction=0.046, pad=0.04)
    
    return im

In [13]:
def visualize_grid_values(grid, feature, fire_ID, fireday, patch, ax=None):

    """
    Generates a scatter plot of a specific fire patch (grid)
    
    :param grid: feature's 2D array
    :param feature: feature to plot
    :param fire_ID: Fire ID
    :param fireday: Fire day
    :param patch: patch index relative to the entire fire
    """

    grid = np.asarray(grid)
    rows, cols = np.where(~np.isnan(grid))
    values = grid[rows, cols]

    if ax is None:
        plt.figure(figsize=(7, 7))
        ax = plt.gca()
    
    sc = ax.scatter(cols, rows, c=values, cmap='hot', s=3)
    ax.set_title(f"{feature} - Fire {fire_ID} day {fireday} patch {patch}")
    ax.set_xlabel("x index")
    ax.set_ylabel("y index")
    return sc

In [14]:
def visualize_grid_features(features_list, features_grid, fire_ID, fireday, patch):

    """
    Generates scatter plot of all features for a specific fire patch (sample)
    
    :param features_list: names of features
    :param grid: features mult-dimenttional grid
    :param fire_ID: Fire ID
    :param fireday: Fire day
    :param patch: patch index relative to the entire fire
    """

    n_features = len(features_list)
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
    
    for idx, (channel, feature) in enumerate(enumerate(features_list)):
        row = idx // n_cols
        col = idx % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
    
        sc = visualize_grid_values(
            features_grid[channel],
            feature,
            fire_ID=fire_ID,
            fireday=fireday,
            patch=patch,
            ax=ax
        )
        plt.colorbar(sc, ax=ax)
    
    # Hide empty subplots
    for idx in range(n_features, n_rows*n_cols):
        row = idx // n_cols
        col = idx % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# LST to UTC conversion

In [15]:
from timezonefinder import TimezoneFinder
import pytz
from datetime import datetime, timedelta

In [16]:
FIRE_SUBSETS_FOLDER = 'Fire_Subsets' 

In [17]:
def local_to_utc(row, tf, lon_col, lat_col):
    """
    Add columns corresponding to start date, end date and noon time in UTC (calculated from local time)
    UTC times are necessary to fetch weather variables from ERA5
    
    :param row: Dataframe row
    :param tf: Timezone finder
    :param lon_col: The longitude column name
    :param lat_col: The latitude column name
    """
    
    if pd.isna(row[lat_col]) or pd.isna(row[lon_col]) or pd.isna(row['DOB']):
        return pd.Series([None, None, None, None])
    
    # Get the timezone at the specified coordinates
    tz_name = tf.timezone_at(lat=row[lat_col], lng=row[lon_col])

    if tz_name is None:
        tz_name = "UTC"

    tz = pytz.timezone(tz_name)

    # Get the start datetime, end datetime, and noon datetime in local time
    date = datetime(int(row['year']), 1, 1) + timedelta(days=int(row['DOB'])-1)
    start_local = tz.localize(datetime(date.year, date.month, date.day))
    end_local = start_local + timedelta(days=1)
    noon_local = tz.localize(datetime(date.year, date.month, date.day, 12))

    # Convert the local times to UTC
    return pd.Series([
        int(date.month),
        start_local.astimezone(pytz.utc).isoformat()[:19],
        end_local.astimezone(pytz.utc).isoformat()[:19],
        noon_local.astimezone(pytz.utc).isoformat()[:19]
    ])

In [19]:
timezone_finder = TimezoneFinder()

# Generate the times for the selected fires

for filename in os.listdir(FIRE_SUBSETS_FOLDER):
    
    file_path = os.path.join(FIRE_SUBSETS_FOLDER, filename)
    
    if os.path.isfile(file_path):
        
        print(f"Processing: {filename}")
        subset_fire_df = pd.read_csv(file_path)
        print(subset_fire_df.shape)

        tqdm.pandas(desc="Calculating Timezones")

        subset_fire_df[['month','start_utc', 'end_utc', 'noon_utc']] = subset_fire_df.progress_apply(
            lambda row: local_to_utc(row, timezone_finder, 'lon', 'lat'),
            axis=1
        )

        subset_fire_df.to_csv(file_path, index=False)

Processing: subset_fire_2024_188.csv
(4863, 25)


Calculating Timezones: 100%|██████████████████████████████████████████████████████| 4863/4863 [00:18<00:00, 268.49it/s]


In [20]:
# Check dimensions after the additions

for filename in os.listdir(FIRE_SUBSETS_FOLDER):
    
    file_path = os.path.join(FIRE_SUBSETS_FOLDER, filename)
    
    if os.path.isfile(file_path):
        
        print(f"Processing: {filename}")
        subset_fire_df = pd.read_csv(file_path)
        print(subset_fire_df.shape)

Processing: subset_fire_2024_188.csv
(4863, 25)


# Topography

## ASTER DEM (Requests)

In [22]:
import xdem
import rasterio
from rasterio.coords import BoundingBox
import json
from copy import deepcopy
import requests
import getpass, pprint, time, os, cgi
import xarray as xr
from pathlib import Path
from pyproj import Transformer
import re

In [23]:
DEM_FOLDER = Path('DEM_API/Canada_DEM')
DEM_FOLDER.mkdir(parents=True, exist_ok=True)
FIRE_SUBSETS_FOLDER = 'Fire_Subsets'

## AppEEARS requests

In [24]:
import os
import json
import numpy as np
from copy import deepcopy
import requests
import getpass, pprint, time, cgi

In [ ]:
# There is no need to run this section as the files are already provided
# To run this section, it is necessary to have an account in NASA AppEEARS https://appeears.earthdatacloud.nasa.gov/

In [25]:
APPEEARS_LOGIN_URL = 'https://appeears.earthdatacloud.nasa.gov/api/login'
APPEEARS_TASK_URL = "https://appeears.earthdatacloud.nasa.gov/api/task"
APPEEARS_LOGIN = 'replace by your username'
APPEEARS_PWD = 'replace by your password'

In [26]:
def generate_canada_dem_requests(
    output_dir,
    step_deg=10,
    lon_min=-142, lon_max=-51,
    lat_min=41, lat_max=84
):
    """
    Generate the requests json files to send to NASA AppEEARS (DEM rasters)
    
    :param output_dir: Output folder path
    :param step_deg: Height/Width of each raster in degrees
    :param lon_min: Starting longitude for the raster
    :param lon_max: Ending longitude for the raster
    :param lat_min: Starting latitude for the raster
    :param lat_max: Ending latitude for the raster
    """

    base_template = {
        "task_type": "area",
        "task_name": "",
        "params": {
            "dates": [{
                "startDate": "07-20-2024",
                "endDate": "07-22-2024",
                "recurring": False,
                "yearRange": [2000, 2050]
            }],
            "layers": [{
                "product": "ASTGTM_NC.003",
                "layer": "ASTER_GDEM_DEM"
            }],
            "geo": {
                "type": "FeatureCollection",
                "features": []
            },
            "output": {
                "format": {"type": "geotiff"},
                "projection": "native",
                "additionalOptions": {}
            }
        }
    }

    for lat in np.arange(lat_min, lat_max, step_deg):
        for lon in np.arange(lon_min, lon_max, step_deg):

            lat2 = min(lat + step_deg, lat_max)
            lon2 = min(lon + step_deg, lon_max)

            # convert to Python native
            lat = float(lat)
            lon = float(lon)
            lat2 = float(lat2)
            lon2 = float(lon2)

            name = f"CAN_lat_{int(lat)}_{int(lat2)}_lon_{int(lon)}_{int(lon2)}"

            polygon = [
                [lon, lat],
                [lon, lat2],
                [lon2, lat2],
                [lon2, lat],
                [lon, lat]
            ]

            req = deepcopy(base_template)
            req["task_name"] = name

            req["params"]["geo"]["features"] = [{
                "type": "Feature",
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [polygon]
                },
                "properties": {}
            }]

            out_path = os.path.join(output_dir, f"{name}.json")

            with open(out_path, "w") as f:
                json.dump(req, f, indent=2)

            print("Created:", out_path)

In [ ]:
REQUEST_FOLDER = 'DEM_API/Canada_DEM/Requests'
Path(REQUEST_FOLDER).mkdir(parents=True, exist_ok=True)

# Generate the requests jsons
generate_canada_dem_requests(
    output_dir=REQUEST_FOLDER,
    step_deg=10
)

In [ ]:
response = requests.post('https://appeears.earthdatacloud.nasa.gov/api/login', auth=(APPEEARS_LOGIN, APPEEARS_PWD))
token_response = response.json()
print(token_response)

In [ ]:
# Submit the requests 

REQUEST_DIR = "DEM_API/Canada_DEM/Requests"

token = token_response["token"]
headers = {"Authorization": f"Bearer {token}"}

submitted = []

print(len(sorted(os.listdir(REQUEST_DIR))))
    
for fname in tqdm(sorted(os.listdir(REQUEST_DIR))):
    if not fname.endswith(".json"):
        continue

    path = os.path.join(REQUEST_DIR, fname)

    with open(path) as f:
        task = json.load(f)

    response = requests.post(
        "https://appeears.earthdatacloud.nasa.gov/api/task",
        json=task,
        headers=headers
    )

    if response.status_code == 200:
        task_response = response.json()
        submitted.append(task_response)
        print("Submitted:", fname, " -- TaskID:", task_response.get("task_id"))
    else:
        print("FAILED:", fname, response.text)

## Preprocessing (with xdem)

In [29]:
def process_dem_folder_xdem(
    dem_original_dir=f"{DEM_FOLDER}/DEM_Files_XDEM/Elevation_original",
    # dem_90m_dir=f"{DEM_FOLDER}/DEM_Files_XDEM/Elevation",
    dem_avg_dir=f"{DEM_FOLDER}/DEM_Files_XDEM/Elevation_average",
    slope_dir=f"{DEM_FOLDER}/DEM_Files_XDEM/Slope",
    aspect_dir=f"{DEM_FOLDER}/DEM_Files_XDEM/Aspect",
    scale_factor=3,
    target_resolution=90,
    resampling="average",
    slope_method="Horn",
):
    """
    Generate the DEM, slope, and aspect folders from the original DEM folders
    
    :param dem_original_dir: Original DEM folder
    :param dem_avg_dir: Target DEM folder (3x3 averaged blocs)
    :param slope_dir: Target slope folder (90m)
    :param aspect_dir: Target aspect folder (90m)
    :param scale_factor: Bloc size
    :param target_resolution: Final resolution
    :param resampling: Resampling method downsampling
    :param slope_method: Slope and aspect calculation algorithm
    """

    dem_original_dir = Path(dem_original_dir)
    # dem_90m_dir = Path(dem_90m_dir)
    dem_avg_dir = Path(dem_avg_dir)
    slope_dir = Path(slope_dir)
    aspect_dir = Path(aspect_dir)

    # Create output folders
    # dem_90m_dir.mkdir(parents=True, exist_ok=True)
    dem_avg_dir.mkdir(parents=True, exist_ok=True)
    slope_dir.mkdir(parents=True, exist_ok=True)
    aspect_dir.mkdir(parents=True, exist_ok=True)

    # Loop over all GeoTIFF DEMs
    for dem_path in dem_original_dir.glob("*.tif"):
        
        print(f"Processing {dem_path.name}")

        # Load DEM
        dem = xdem.DEM(dem_path)

        # Explicitly handle NoData to stop the -32768 warning
        if dem.nodata is None:
            dem.set_nodata(-9999.0)

        # Average 3x3 blocs for DEM
        # Hager: you don't specify a CRS here, so this still remains in is original CRS (EPSG:4326) - degrees
        # Hager: so this is 3x coarser in geographic degrees not meters
        dem_avg = dem.reproject(
            res=dem.res[0] * scale_factor,
            resampling=resampling,
        )

        # Reproject DEM to 90m and to meters coordinates
        # Meters coordinates are necessary to calculate slope and aspect
        # x, y, and z must be in the same unit
        # x and y are coordinates
        # z are elevation values
        # Hager: why do you use two objects: dem_90m_with_m for slope and aspect but dem_90m for the dem?
        # Hager: CRS has other params such as transforms, origin, etc, did you check those?
        # Hager: Can you try the following code snippet:
        
        # Reproject to 90 m
        dem_90m_with_m = dem.reproject(
            crs="EPSG:3347", 
            res=target_resolution, 
            resampling=resampling,
            nodata=dem.nodata
        )


        # Output filenames
        # dem_out = dem_90m_dir / f"{dem_path.stem}_90m{dem_path.suffix}"
        dem_out = dem_avg_dir / f"{dem_path.stem}_avg{dem_path.suffix}"
        slope_out = slope_dir / f"{dem_path.stem.replace('DEM', 'SLOPE')}_90m{dem_path.suffix}"
        aspect_out = aspect_dir / f"{dem_path.stem.replace('DEM', 'ASPECT')}_90m{dem_path.suffix}"

        # Save DEM
        dem_avg.to_file(dem_out, driver="GTiff")

        # Compute terrain attributes
        slope = xdem.terrain.slope(dem_90m_with_m, surface_fit=slope_method)
        slope.to_file(slope_out, driver="GTiff")
        print('Slope Done')
        
        aspect = xdem.terrain.aspect(dem_90m_with_m)
        aspect.to_file(aspect_out, driver="GTiff")
        print('Aspect Done')

        print(f"DEM {dem_path} processed successfully.")

    print("All DEMs processed successfully.")

In [ ]:
%%time
process_dem_folder_xdem()

# process_dem_folder_xdem(
#     dem_original_dir="../DEM_API/DEM_Tiles",
#     dem_avg_dir="../DEM_API/Elevation_average",
#     slope_dir="../DEM_API/Slope",
#     aspect_dir="../DEM_API/Aspect",
#     scale_factor=3,
#     target_resolution=90,
#     resampling="average",
#     slope_method="Horn",
# )

## Pipeline (with xdem)

In [25]:
def get_overlapping_tiles(csv_min_lon, csv_max_lon, csv_min_lat, csv_max_lat, raster_folder):

    """
    Get all the DEM rasters necessary for a fire based on longitude and latitudes bounds for the fire
    
    :param csv_min_lon: Fire minimal longitude
    :param csv_max_lon: Fire maximal longitude
    :param csv_min_lat: Fire minimal latitude
    :param csv_max_lat: Fire maximal longitude
    :param raster_folder: Ratser folder
    """

    raster_paths = list(Path(raster_folder).glob("*.tif"))
    overlapping = []

    for rpath in raster_paths:
        # Example filename: DEM_lat_51_61_lon_-132_-122_90m.tif
        fname = rpath.stem 
        # Extract the min and max lon and lat
        m = re.search(r"lat_(-?\d+)_(-?\d+)_lon_(-?\d+)_(-?\d+)", fname)
        if not m:
            continue
        min_lat, max_lat, min_lon, max_lon = map(int, m.groups())

        # Check if CSV bounding box overlaps raster bounding box
        if (csv_max_lon >= min_lon and csv_min_lon <= max_lon and
            csv_max_lat >= min_lat and csv_min_lat <= max_lat):
            # overlapping.append(rpath.name)
            overlapping.append(rpath)

    return overlapping

In [26]:
# Quick verification
get_overlapping_tiles(csv_min_lon=-125, csv_max_lon=-126, csv_min_lat=58, csv_max_lat=59, raster_folder='DEM_API/Canada DEM/DEM files XDEM/Elevation')

[]

In [30]:
def sample_raster_for_points(raster_path, coords, output_array):
        
    """
    Get the values (DEM, slope, or aspect) from a specified raster corresponding 
    to the specidied coordinates
    
    :param raster_path: Raster path
    :param coords: List of coordinates
    :param output_array: values array
    """

    with rasterio.open(raster_path) as src:
        bounds = src.bounds
        
        # Mask points inside raster bounds
        mask = [(x >= bounds.left and x <= bounds.right and
                    y >= bounds.bottom and y <= bounds.top) for x, y in coords]
        
        if any(mask):
            coords_in = [c for c, m in zip(coords, mask) if m]
            vals = np.array([v[0] for v in src.sample(coords_in)], dtype=float)
            
            # Handle nodata
            if src.nodata is not None:
                vals[vals == src.nodata] = np.nan
            
            # Assign sampled values back to correct positions
            j = 0
            for i, m in enumerate(mask):
                if m:
                    output_array[i] = vals[j]
                    j += 1

In [31]:
def add_dem_slope_aspect_bulk(df,
                              dem_paths,
                              slope_paths,
                              aspect_paths,
                              lat_col="lat",
                              lon_col="lon",
                              x_col="easting",
                              y_col="northing"):
    """
    Sample DEM, slope, and aspect from multiple raster files.
    For each point, it uses the raster tile that contains it.
    Points not inside any raster are set to NaN.
    
    :param df: dataframe 
    :param dem_paths: List of DEM raster paths necessary for the fire
    :param slope_paths: List of slope raster paths necessary for the fire
    :param aspect_paths: List of aspect raster paths necessary for the fire
    :param lat_col: Name of the latitude column in the dataframe
    :param lon_col: Name of the longitude column in the dataframe
    :param x_col: Name of the meter x column in the dataframe
    :param y_col: Name of the meter y column in the dataframe
    """

    x_src = df[x_col]
    y_src = df[y_col]

    # Latitude/Longitude coordinates
    coords_lonlat = list(zip(df[lon_col].values, df[lat_col].values))
    # Meter coordinates
    coords_xy = list(zip(x_src, y_src))

    # Initialize output arrays
    dem_vals = np.full(len(df), np.nan, dtype=float)
    slope_vals = np.full(len(df), np.nan, dtype=float)
    aspect_vals = np.full(len(df), np.nan, dtype=float)

    # Sample DEM tiles (lat/lon)
    for dem_path in dem_paths:
        sample_raster_for_points(dem_path, coords_lonlat, dem_vals)
    # Sample Slope tiles (m)
    for slope_path in slope_paths:
        sample_raster_for_points(slope_path, coords_xy, slope_vals)
    # Sample Aspect tiles (m)
    for aspect_path in aspect_paths:
        sample_raster_for_points(aspect_path, coords_xy, aspect_vals)

    # Assign to DataFrame
    df["dem_v2"] = dem_vals
    df["slope_v2"] = slope_vals
    df["aspect_v2"] = aspect_vals

    return df

In [32]:
DEM_FOLDER = "DEM_API/Canada_DEM"
ELEVATION_FOLDER = f"{DEM_FOLDER}/DEM_Files_XDEM/Elevation_average"
SLOPE_FOLDER = f"{DEM_FOLDER}/DEM_Files_XDEM/Slope"
ASPECT_FOLDER = f"{DEM_FOLDER}/DEM_Files_XDEM/Aspect"

In [ ]:
%%time

FIRE_SUBSETS_FOLDER = 'Fire_Subsets' 

for csv_file in Path(FIRE_SUBSETS_FOLDER).glob("*.csv"):
    
    print(f"Processing {csv_file.name}...")
    df = pd.read_csv(csv_file)

    # Get bounding box
    min_lon, max_lon = df["lon"].min(), df["lon"].max()
    min_lat, max_lat = df["lat"].min(), df["lat"].max()

    print(min_lon, max_lon)
    print(min_lat, max_lat)

    # Get overlapping DEM tiles
    dem_tiles = get_overlapping_tiles(min_lon, max_lon, min_lat, max_lat, ELEVATION_FOLDER)
    slope_tiles = get_overlapping_tiles(min_lon, max_lon, min_lat, max_lat, SLOPE_FOLDER)
    aspect_tiles = get_overlapping_tiles(min_lon, max_lon, min_lat, max_lat, ASPECT_FOLDER)
    print(dem_tiles)
    print(slope_tiles)
    print(aspect_tiles)

    if not dem_tiles:
        print(f"No DEM tile found for {csv_file.name}, skipping...")
        continue

    df_aug = add_dem_slope_aspect_bulk(
        df,
        dem_paths=dem_tiles,
        slope_paths=slope_tiles,
        aspect_paths=aspect_tiles,
        lat_col="lat",
        lon_col="lon",
        x_col="easting",
        y_col="northing"
    )

    # Save CSV
    df_aug.to_csv(Path(FIRE_SUBSETS_FOLDER)/csv_file.name, index=False)

    del df, df_aug

    print('\n')

In [ ]:
for filename in os.listdir(FIRE_SUBSETS_FOLDER):
    
    # Construct the full path
    file_path = os.path.join(FIRE_SUBSETS_FOLDER, filename)
    
    if os.path.isfile(file_path):
        
        print(f"Processing: {filename}")
        subset_fire_df = pd.read_csv(file_path)
        print(subset_fire_df.shape)

In [ ]:
# Quick verification
subset_fire_df = pd.read_csv(f'{FIRE_SUBSETS_FOLDER}/subset_fire_2024_188.csv')
subset_fire_df[subset_fire_df['fireday']==6][['DOB', 'dem', 'dem_v2', 'slope', 'slope_v2', 'aspect', 'aspect_v2']]

## Preprocessing (GDAL)

In [ ]:
# This an alternative method for topography calculation

In [33]:
import subprocess
from pathlib import Path
import rasterio
from rasterio.coords import BoundingBox
import json
from copy import deepcopy
import requests
import getpass, pprint, time, os, cgi
import xarray as xr
from pyproj import Transformer
import re
import os
import sys

In [34]:
def process_dem_folder_gdal(
    dem_original_dir,
    dem_avg_dir,
    dem_90m_dir,
    slope_dir,
    aspect_dir,
    target_resolution=90,
    scale_factor=3,
    resampling="cubicspline",
    slope_method="ZevenbergenThorne"
):
    """
    Generate the DEM, slope, and aspect folders from the original DEM folders using GDAL
    # Hager: did you try other methods for the slope? did you verify if it is the same method they used?
    """

    # Dynamic Path Setup for OSGeo4W
    gdal_bin = Path(os.environ.get("USERPROFILE", "")) / "AppData/Local/Programs/OSGeo4W/bin"
    
    if not gdal_bin.exists():
        # Fallback to standard C drive install if AppData fails
        gdal_bin = Path("C:/OSGeo4W/bin")

    env = os.environ.copy()
    env["PATH"] = f"{gdal_bin}{os.pathsep}{env['PATH']}"
    
    # Define paths
    src_dir = Path(dem_original_dir)
    out_dem_dir = Path(dem_90m_dir)
    out_slope_dir = Path(slope_dir)
    out_aspect_dir = Path(aspect_dir)
    out_dem_avg_dir = Path(dem_avg_dir)

    # Create directories
    for d in [out_dem_dir, out_slope_dir, out_aspect_dir, out_dem_avg_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # Identify tool paths
    gdalwarp = str(gdal_bin / "gdalwarp.exe")
    gdaldem = str(gdal_bin / "gdaldem.exe")

    # Loop through files
    for dem_path in src_dir.glob("*.tif"):

        if dem_path.name != 'DEM_lat_49_50_lon_-63_-62.tif':
            continue

        print(dem_path.name)
        
        with rasterio.open(dem_path) as src:
            orig_res_x, orig_res_y = src.res 
            target_res_x = orig_res_x * scale_factor
            target_res_y = orig_res_y * scale_factor
    
        print(f"\n--- Processing: {dem_path.name} ---")

        # Define Output Filenames
        dem_90m_out = out_dem_dir / f"{dem_path.stem}_90m.tif"
        dem_avg_out = out_dem_avg_dir / f"{dem_path.stem}_AVG.tif"
        slope_out = out_slope_dir / f"{dem_path.stem.replace('DEM', 'SLOPE')}_90m.tif"
        aspect_out = out_aspect_dir / f"{dem_path.stem.replace('DEM', 'ASPECT')}_90m.tif"

        try:
            # Reproject DEM to EPSG:3347 & Resample to 90m (for slope and aspect calculation)
            print(f"  > Reprojecting to 90m (EPSG:3347)...")
            subprocess.run([
                gdalwarp, 
                "-t_srs", "EPSG:3347", 
                "-tr", str(target_resolution), str(target_resolution),
                "-r", resampling, 
                "-tap", 
                "-overwrite", 
                str(dem_path), str(dem_90m_out)
            ], check=True, env=env)

            # Average DEM 3 by 3 pixels (for DEM calculations)
            print(f"  > Averaging DEM...")
            subprocess.run([
                gdalwarp, 
                "-tr", str(target_res_x), str(target_res_y), 
                "-r", "average", 
                "-overwrite", 
                str(dem_path), str(dem_avg_out)
            ], check=True, env=env)

            # Calculate Slope
            print(f"  > Calculating Slope ({slope_method})...")
            subprocess.run([
                gdaldem, "slope", 
                str(dem_90m_out), str(slope_out), 
                "-alg", slope_method, 
                "-compute_edges"
            ], check=True, env=env)

            # Calculate Aspect
            print(f"  > Calculating Aspect...")
            subprocess.run([
                gdaldem, "aspect", 
                str(dem_90m_out), str(aspect_out), 
                "-alg", slope_method, 
                "-compute_edges", 
                "-zero_for_flat"
            ], check=True, env=env)

            print(f"Successfully processed {dem_path.name}")

        except subprocess.CalledProcessError as e:
            print(f"ERROR: GDAL failed on {dem_path.name}. Specific error: {e}")
        except FileNotFoundError:
            print(f"ERROR: Could not find GDAL at {gdal_bin}. Please check your OSGeo4W installation.")
            return

    print("\nAll files processed successfully.")

In [ ]:
%%time

process_dem_folder_gdal(
    dem_original_dir=f"{DEM_FOLDER}/DEM_Files_GDAL/Elevation_original",
    dem_avg_dir=f"{DEM_FOLDER}/DEM_Files_GDAL/Elevation_average",
    dem_90m_dir=f"{DEM_FOLDER}/DEM_Files_GDAL/Elevation",
    slope_dir=f"{DEM_FOLDER}/DEM_Files_GDAL/Slope",
    aspect_dir=f"{DEM_FOLDER}/DEM_Files_GDAL/Aspect",
    target_resolution=90,
    scale_factor=3,
    resampling="cubicspline", # gave better results for slope and aspect
    # resampling="average",
    slope_method="ZevenbergenThorne"
)

# process_dem_folder_gdal(
#     dem_original_dir="../DEM_API/DEM_Tiles",
#     dem_avg_dir="../DEM_API/Elevation_average",
#     dem_90m_dir="../DEM_API/Elevation",
#     slope_dir="../DEM_API/Slope",
#     aspect_dir="../DEM_API/Aspect",
#     target_resolution=90,
#     scale_factor=3,
#     resampling="cubicspline", # gave better results for slope and aspect
#     # resampling="average",
#     slope_method="ZevenbergenThorne"
# )

## DEM pipeline (with GDAL)

In [35]:
from scipy.ndimage import uniform_filter

In [36]:
DEM_FOLDER = "DEM_API/Canada_DEM"
ELEVATION_FOLDER = f"{DEM_FOLDER}/DEM_Files_GDAL/Elevation"
ELEVATION_AVG_FOLDER = f"{DEM_FOLDER}/DEM_Files_GDAL/Elevation_average"
SLOPE_FOLDER = f"{DEM_FOLDER}/DEM_Files_GDAL/Slope"
ASPECT_FOLDER = f"{DEM_FOLDER}/DEM_Files_GDAL/Aspect"

In [37]:
def get_overlapping_tiles(csv_min_lon, csv_max_lon, csv_min_lat, csv_max_lat, raster_folder):

    raster_paths = list(Path(raster_folder).glob("*.tif"))
    overlapping = []

    for rpath in raster_paths:
        # Example filename: DEM_lat_51_61_lon_-132_-122_90m.tif
        fname = rpath.stem
        m = re.search(r"lat_(-?\d+)_(-?\d+)_lon_(-?\d+)_(-?\d+)", fname)
        if not m:
            continue
        min_lat, max_lat, min_lon, max_lon = map(int, m.groups())

        # Check if CSV bounding box overlaps raster bounding box
        if (csv_max_lon >= min_lon and csv_min_lon <= max_lon and
            csv_max_lat >= min_lat and csv_min_lat <= max_lat):
            overlapping.append(rpath)

    return overlapping

In [46]:
def sample_rasters_multi_tile(df, tile_paths, mode='direct', size=3, col_name="val"):
    
    """
    Get topography values from rasters
    mode='mean': calculates nxn mean (use for DEM)
    mode='direct': samples exact pixel (use for your 90m DEM/Slope/Aspect)
    
    :param df: Dataframe
    :param tile_paths: list of rasters
    :param mode: 'mean' and 'direct'
    :param size: Bloc size (3 for 3x3 aggregation) 
    :param col_name: Column name
    """

    col_name = f'{col_name}_v2'
    df[col_name] = np.nan
    
    if not tile_paths:
        return df

    for rpath in tile_paths:
        with rasterio.open(rpath) as src:
            # Handle Coordinate Systems
            # If raster is 3347, transform CSV lat/lon. If 4326, use as is.
            if src.crs.to_string() == "EPSG:3347":
                transformer = Transformer.from_crs("EPSG:4326", "EPSG:3347", always_xy=True)
                points_x, points_y = transformer.transform(df['lon'].values, df['lat'].values)
            else:
                points_x, points_y = df['lon'].values, df['lat'].values

            # Only process points inside this tile
            bounds = src.bounds
            mask = (points_x >= bounds.left) & (points_x <= bounds.right) & \
                   (points_y >= bounds.bottom) & (points_y <= bounds.top)
            
            if not mask.any():
                continue

            # Read Data
            data = src.read(1).astype('float32')
            
            if src.nodata is not None:
                data[data == src.nodata] = np.nan

            if mode == 'mean':
                # Ensure data is float for decimal precision
                data = data.astype('float32')
                
                # Create a mask: 1.0 for valid data, 0.0 for NaNs
                mask = np.where(np.isnan(data), 0, 1).astype('float32')                
                # Fill NaNs with 0
                data_zeroed = np.nan_to_num(data, nan=0.0)
                
                # Calculate the sum of values in the 3x3 window
                v_sum = uniform_filter(data_zeroed, size=size, mode='reflect') * (size**2)
                
                # Calculate the count of valid pixels in the 3x3 window
                v_count = uniform_filter(mask, size=size, mode='reflect') * (size**2)
                
                data = np.divide(v_sum, v_count, out=np.full_like(data, np.nan), where=v_count > 0)

            # Vectorized Indexing
            rows, cols = rasterio.transform.rowcol(src.transform, points_x[mask], points_y[mask])
            
            # Clip to prevent index errors at the very edge of tiles
            rows = np.clip(rows, 0, data.shape[0] - 1)
            cols = np.clip(cols, 0, data.shape[1] - 1)
            
            df.loc[mask, col_name] = data[rows, cols]

    return df

In [41]:
%%time

FIRE_SUBSETS_FOLDER = 'Fire_Subsets' 

for csv_file in Path(FIRE_SUBSETS_FOLDER).glob("*.csv"):

    if csv_file.name != 'subset_fire_2024_188.csv':
        continue

    print(f'Processing {csv_file} ...')
    
    df = pd.read_csv(csv_file)
    
    # Identify Tiles
    min_lon, max_lon = df["lon"].min(), df["lon"].max()
    min_lat, max_lat = df["lat"].min(), df["lat"].max()
    
    # dem_tiles = get_overlapping_tiles(min_lon, max_lon, min_lat, max_lat, ELEVATION_FOLDER)
    
    dem_tiles = get_overlapping_tiles(min_lon, max_lon, min_lat, max_lat, ELEVATION_AVG_FOLDER)
    slope_tiles = get_overlapping_tiles(min_lon, max_lon, min_lat, max_lat, SLOPE_FOLDER)
    aspect_tiles = get_overlapping_tiles(min_lon, max_lon, min_lat, max_lat, ASPECT_FOLDER)

    # DEM: using the averaged tiles
    print("Sampling DEM...")
    df = sample_rasters_multi_tile(df, dem_tiles, mode='direct', col_name="dem")
    
    # Slope: using the 90m EPSG:3347 tiles
    print("Sampling Slope...")
    df = sample_rasters_multi_tile(df, slope_tiles, mode='direct', col_name="slope")

    # Aspect: using the 90m EPSG:3347 tiles
    print("Sampling Aspect...")
    df = sample_rasters_multi_tile(df, aspect_tiles, mode='direct', col_name="aspect")

    df.to_csv(Path(FIRE_SUBSETS_FOLDER)/csv_file.name, index=False)

    del df

Processing Fire_Subsets\subset_fire_2024_188.csv ...
Sampling DEM...
Sampling Slope...
Sampling Aspect...
CPU times: total: 312 ms
Wall time: 358 ms


In [47]:
for filename in os.listdir(FIRE_SUBSETS_FOLDER):

    file_path = os.path.join(FIRE_SUBSETS_FOLDER, filename)
    
    if os.path.isfile(file_path):
        
        print(f"Processing: {filename}")
        subset_fire_df = pd.read_csv(file_path)
        print(subset_fire_df.shape)

Processing: subset_fire_2024_188.csv
(4863, 28)


In [43]:
def calculate_relative_error(df, column_pairs):
    """
    Calculate relative errors
    
    :param df: dataframe
    :param column_pairs: list of pair of variables
    """
    error_df = pd.DataFrame(index=df.index)
    
    for actual_col, pred_col in column_pairs:
        error_name = f"rel_error_{pred_col}"
        
        valid_mask = df[actual_col].notna() & df[pred_col].notna()
        
        # Initialize the column with NaN
        error_df[error_name] = np.nan
        
        error_df.loc[valid_mask, error_name] = np.where(
            (df.loc[valid_mask, actual_col] == 0) & (df.loc[valid_mask, pred_col] == 0), 
            0.0,
            np.where(
                (df.loc[valid_mask, actual_col] == 0) & (df.loc[valid_mask, pred_col] > 0), 
                np.nan,
                ((df.loc[valid_mask, pred_col] - df.loc[valid_mask, actual_col]).abs() / df.loc[valid_mask, actual_col].abs()) * 100
            )
        )
        
    return error_df

In [51]:
def calculate_slope_summary(y_true, y_pred):
    """
    Calculates Slope metrics and returns a summary DataFrame
    including MAE, RMSE, and error extremes.
    """

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Remove any NaN pairs
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    t = y_true[mask]
    p = y_pred[mask]
    
    if len(t) == 0:
        return pd.DataFrame([{"Error": "No valid data points"}])

    # Calculate Linear Errors
    errors = np.abs(t - p)
    
    # Statistical Metrics
    mae = np.mean(errors)
    rmse = np.sqrt(np.mean(errors**2))
    max_err = np.max(errors)
    std_err = np.std(t - p)
    median_err = np.median(errors)

    summary_df = pd.DataFrame({
        "Metric": ["MAE", "RMSE", "Median Error", "Std Dev", "Sample Count"],
        "Value": [mae, rmse, median_err, std_err, len(t)]
    })

    return summary_df.round(4)

In [53]:
def calculate_aspect_summary(y_true, y_pred):
    """
    Calculates circular Aspect metrics and returns a summary DataFrame
    including MAE, RMSE, and error extremes.
    """
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    t = y_true[mask]
    p = y_pred[mask]
    
    if len(t) == 0:
        return pd.DataFrame([{"Error": "No valid data points"}])

    # Circular Difference Logic (Shortest distance around 360)
    diff = np.abs(t - p) % 360
    circular_errors = np.where(diff > 180, 360 - diff, diff)

    # Calculate Statistics
    mae = np.mean(circular_errors)
    rmse = np.sqrt(np.mean(circular_errors**2))
    max_err = np.max(circular_errors)
    std_err = np.std(circular_errors)
    median_err = np.median(circular_errors)

    summary_df = pd.DataFrame({
        "Metric": ["MAE", "RMSE", "Median Error", "Std Dev", "Sample Count"],
        "Value": [mae, rmse, median_err, std_err, len(t)]
    })

    return summary_df.round(4)

## GDAL without disjointed tiles (Final Method)

In [54]:
import subprocess
from pathlib import Path
import rasterio
from rasterio.coords import BoundingBox
import json
from copy import deepcopy
import requests
import getpass, pprint, time, os, cgi
import xarray as xr
from pyproj import Transformer
import re
import os
import sys

In [55]:
def generate_fire_terrain(
    fire_id, 
    fire_df, 
    vrt_path, 
    out_dem_dir, 
    out_slope_dir, 
    out_aspect_dir, 
    out_dem_avg_dir,
    buffer_deg=1.0,
    base_res=90,
    scale_factor=3
):
    """
    Crops a DEM from a VRT for a specific fire, and calculates Slope, Aspect, and a 3x3 Averaged DEM.
    """
    # Create directories
    for directory in [out_dem_dir, out_slope_dir, out_aspect_dir, out_dem_avg_dir]:
        Path(directory).mkdir(parents=True, exist_ok=True)

    # Buffered Lat/Lon bounding box
    min_lon = fire_df['lon'].min() - buffer_deg
    max_lon = fire_df['lon'].max() + buffer_deg
    min_lat = fire_df['lat'].min() - buffer_deg
    max_lat = fire_df['lat'].max() + buffer_deg

    # Convert the Lat/Lon box to EPSG:3347 Meters
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:3347", always_xy=True)
    min_x_m, min_y_m = transformer.transform(min_lon, min_lat)
    max_x_m, max_y_m = transformer.transform(max_lon, max_lat)

    # Define all paths
    out_dem_path = str(Path(out_dem_dir) / f"{fire_id}_dem_{base_res}m.tif")
    out_slope_path = str(Path(out_slope_dir) / f"{fire_id}_slope_{base_res}m.tif")
    out_aspect_path = str(Path(out_aspect_dir) / f"{fire_id}_aspect_{base_res}m.tif")
    
    # Average
    out_dem_avg_path = str(Path(out_dem_avg_dir) / f"{fire_id}_dem_avg.tif")

    # Run GDAL 
    try:
        # STEP 1: Crop and project standard DEM from VRT
        print(f"[{fire_id}] 1/4: Cropping and projecting DEM to {base_res}m...")
        subprocess.run([
            "gdalwarp", 
            "-t_srs", "EPSG:3347",
            "-te", str(min_x_m), str(min_y_m), str(max_x_m), str(max_y_m),
            "-tr", str(base_res), str(base_res),
            "-tap", 
            "-r", "cubicspline",
            "-srcnodata", "-9999", 
            "-dstnodata", "-9999",
            "-overwrite",
            str(vrt_path), out_dem_path   
        ], check=True, capture_output=True, text=True)

        # STEP 2: Create the 3x3 Averaged DEM 
        print(f"[{fire_id}] 2/4: Averaging DEM 3x3...")
        
        # Determine the raw resolution of the VRT in degrees
        with rasterio.open(vrt_path) as src:
            orig_res_x, orig_res_y = src.res 
            # print(orig_res_x, orig_res_y)
            target_res_x = orig_res_x * scale_factor
            target_res_y = orig_res_y * scale_factor

        # Crop and Average the raw Lat/Lon data
        subprocess.run([
            "gdalwarp", 
            "-te", str(min_lon), str(min_lat), str(max_lon), str(max_lat), # Crop in degrees
            "-tr", str(target_res_x), str(target_res_y),                   # Scale in degrees
            "-r", "average", 
            "-ot", "Float32",
            "-tap",            # Snaps the grid globally
            "-overwrite", 
            str(vrt_path), out_dem_avg_path                                # Raw source to Raw output
        ], check=True, capture_output=True, text=True)

        # STEP 3: Calculate Slope
        print(f"[{fire_id}] 3/4: Calculating Slope...")
        subprocess.run([
            "gdaldem", "slope", 
            out_dem_path, out_slope_path,     
            # "-alg", "ZevenbergenThorne", 
            "-alg", "Horn",
            "-compute_edges"              
        ], check=True, capture_output=True, text=True)

        # STEP 4: Calculate Aspect
        print(f"[{fire_id}] 4/4: Calculating Aspect...")
        subprocess.run([
            "gdaldem", "aspect", 
            out_dem_path, out_aspect_path,
            # "-alg", "ZevenbergenThorne", 
            "-alg", "Horn",
            "-compute_edges",
            "-zero_for_flat"
        ], check=True, capture_output=True, text=True)
        
        print(f"[{fire_id}] SUCCESS!.\n")

    except subprocess.CalledProcessError as e:
        
        print(f"[{fire_id}] ERROR: GDAL FAILED. Here is the exact error message:")
        print(e.stderr)

In [56]:
# target_ids = ['2024_188', '2024_306', '2024_486', '2024_545', '2024_560']
target_ids = ['2024_188']

for target_fire_id in tqdm(target_ids):

    generate_fire_terrain(
        fire_id=target_fire_id, 
        fire_df=fire_growth_2024[fire_growth_2024['ID']==target_fire_id], 
        vrt_path='DEM_API/raw_mosaic.vrt', 
        out_dem_dir='DEM_API/Elevation', 
        out_slope_dir='DEM_API/Slope',
        out_aspect_dir='DEM_API/Aspect',
        out_dem_avg_dir='DEM_API/Elevation_avg',
        buffer_deg=1.0,
        base_res=90,
        scale_factor=3
    )

    # generate_fire_terrain(
    #     fire_id=target_fire_id, 
    #     fire_df=fire_growth_2024[fire_growth_2024['ID']==target_fire_id], 
    #     vrt_path='../DEM_API/raw_mosaic.vrt', 
    #     out_dem_dir='../DEM_API/Elevation', 
    #     out_slope_dir='../DEM_API/Slope',
    #     out_aspect_dir='../DEM_API/Aspect',
    #     out_dem_avg_dir='../DEM_API/Elevation_avg',
    #     buffer_deg=1.0,
    #     base_res=90,
    #     scale_factor=3
    # )

  0%|                                                                                            | 0/1 [00:00<?, ?it/s]

[2024_188] 1/4: Cropping and projecting DEM to 90m...
[2024_188] 2/4: Averaging DEM 3x3...
[2024_188] 3/4: Calculating Slope...
[2024_188] 4/4: Calculating Aspect...


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:19<00:00, 19.35s/it]

[2024_188] SUCCESS!.



## GDAL with VRT (Tests)

In [59]:
from scipy.ndimage import uniform_filter, map_coordinates

In [57]:
# DEM_FOLDER = "../DEM_API"
DEM_FOLDER = "DEM_API"
ELEVATION_FOLDER = f"{DEM_FOLDER}/Elevation"
ELEVATION_AVG_FOLDER = f"{DEM_FOLDER}/Elevation_avg"
SLOPE_FOLDER = f"{DEM_FOLDER}/Slope"
ASPECT_FOLDER = f"{DEM_FOLDER}/Aspect"

In [60]:
def get_tiles(fire_id, raster_folder):
    """
    Retrieves the specific terrain tile(s) for a given fire ID from a folder.
    """
    # Use glob to instantly find any .tif file that starts with the fire_id
    # e.g., "2024_545_*.tif"
    overlapping = list(Path(raster_folder).glob(f"{fire_id}_*.tif"))
    
    return overlapping

In [61]:
get_tiles(fire_id='2024_188', raster_folder=SLOPE_FOLDER)

[WindowsPath('../DEM_API/Slope/2024_188_slope_90m.tif')]

In [64]:
def sample_rasters_multi_tile(df, tile_paths, mode='direct', size=3, col_name="val_v2"):
    """
    Get topography values from rasters
    mode='mean': calculates nxn mean
    mode='direct': samples exact pixel 
    """

    col_name = f'{col_name}_v2'
    df[col_name] = np.nan
    
    if not tile_paths:
        return df

    for rpath in tile_paths:
        with rasterio.open(rpath) as src:
            # Handle Coordinate Systems
            if src.crs and src.crs.to_string() == "EPSG:3347":
                transformer = Transformer.from_crs("EPSG:4326", "EPSG:3347", always_xy=True)
                points_x, points_y = transformer.transform(df['lon'].values, df['lat'].values)
            else:
                points_x, points_y = df['lon'].values, df['lat'].values

            # Only process points inside this tile
            bounds = src.bounds
            mask = (points_x >= bounds.left) & (points_x <= bounds.right) & \
                   (points_y >= bounds.bottom) & (points_y <= bounds.top)
            
            if not mask.any():
                continue

            # Read Data
            data = src.read(1).astype('float32')
            
            if src.nodata is not None:
                data[data == src.nodata] = np.nan

            if mode == 'mean':
                data = data.astype('float32')
                
                valid_pixel_mask = np.where(np.isnan(data), 0, 1).astype('float32')                
                
                data_zeroed = np.nan_to_num(data, nan=0.0)
                v_sum = uniform_filter(data_zeroed, size=size, mode='reflect') * (size**2)
                
                # Use the new variable name here
                v_count = uniform_filter(valid_pixel_mask, size=size, mode='reflect') * (size**2)
                
                data = np.divide(v_sum, v_count, out=np.full_like(data, np.nan), where=v_count > 0)

            # Vectorized Indexing
            rows, cols = rasterio.transform.rowcol(src.transform, points_x[mask], points_y[mask])
            
            rows = np.clip(rows, 0, data.shape[0] - 1)
            cols = np.clip(cols, 0, data.shape[1] - 1)
            
            df.loc[mask, col_name] = data[rows, cols]

    return df

In [65]:
%%time

FIRE_SUBSETS_FOLDER = 'Fire_Subsets' 

for csv_file in Path(FIRE_SUBSETS_FOLDER).glob("*.csv"):

    print(f'Processing {csv_file} ...')
    
    df = pd.read_csv(csv_file)
    fire_id = df['ID'].values[0]
    print(fire_id)
    
    dem_tiles = get_tiles(fire_id, raster_folder=ELEVATION_AVG_FOLDER)
    print(dem_tiles)
    slope_tiles = get_tiles(fire_id, raster_folder=SLOPE_FOLDER)
    print(slope_tiles)
    aspect_tiles = get_tiles(fire_id, raster_folder=ASPECT_FOLDER)
    print(aspect_tiles)

    # Elevation: Using the averaged (3x3) Lat/Lon tiles
    df = sample_rasters_multi_tile(df, dem_tiles, mode='direct', col_name="dem")

    # Slope: using the 90m EPSG:3347 tiles
    print("Sampling Slope...")
    df = sample_rasters_multi_tile(df, slope_tiles, mode='direct', col_name="slope")

    # Aspect: using the 90m EPSG:3347 tiles
    print("Sampling Aspect...")
    df = sample_rasters_multi_tile(df, aspect_tiles, mode='direct', col_name="aspect")

    df.to_csv(Path(FIRE_SUBSETS_FOLDER)/csv_file.name, index=False)

    del df

Processing Fire_Subsets\subset_fire_2024_188.csv ...
2024_188
[WindowsPath('../DEM_API/Elevation_avg/2024_188_dem_avg.tif')]
[WindowsPath('../DEM_API/Slope/2024_188_slope_90m.tif')]
[WindowsPath('../DEM_API/Aspect/2024_188_aspect_90m.tif')]
Sampling Slope...
Sampling Aspect...
CPU times: total: 703 ms
Wall time: 815 ms


In [66]:
for filename in os.listdir(FIRE_SUBSETS_FOLDER):

    file_path = os.path.join(FIRE_SUBSETS_FOLDER, filename)
    
    if os.path.isfile(file_path):
        
        print(f"Processing: {filename}")
        subset_fire_df = pd.read_csv(file_path)
        print(subset_fire_df.shape)

Processing: subset_fire_2024_188.csv
(4863, 28)


In [67]:
all_dfs = []

for filename in os.listdir(FIRE_SUBSETS_FOLDER):
    
    file_path = os.path.join(FIRE_SUBSETS_FOLDER, filename)
    
    if os.path.isfile(file_path) and filename.endswith('.csv'):
        print(f"Reading: {filename}")
        
        # Read the file
        subset_df = pd.read_csv(file_path)
        
        subset_df['source_file'] = filename
        
        all_dfs.append(subset_df)

fire_growth_combined = pd.concat(all_dfs, axis=0, ignore_index=True)

print(f"Total rows: {len(fire_growth_combined)}")

Reading: subset_fire_2024_188.csv
Total rows: 4863


In [68]:
pairs = [('dem', 'dem_v2'), ('slope', 'slope_v2'), ('aspect', 'aspect_v2')]
results = calculate_relative_error(fire_growth_combined, pairs)
summary_stats = results.describe(percentiles=[0.5])
summary_stats_rounded = summary_stats.round(6)
summary_stats_rounded

,rel_error_dem_v2,rel_error_slope_v2,rel_error_aspect_v2
count,4863.000000,4863.000000,4863.000000
mean,0.674920,20.514650,103.550789
std,0.629157,20.053534,3357.317172
min,0.000253,0.000229,0.001034
50%,0.492256,14.919269,3.213317
max,5.322614,212.972070,183312.689363


In [69]:
calculate_slope_summary(fire_growth_combined['dem'], fire_growth_combined['dem_v2'])

,Metric,Value
0,MAE,8.1061
1,RMSE,11.5044
2,Median Error,5.6057
3,Std Dev,11.1850
4,Sample Count,4863.0000


In [70]:
calculate_slope_summary(fire_growth_combined['slope'], fire_growth_combined['slope_v2'])

,Metric,Value
0,MAE,2.9986
1,RMSE,3.9837
2,Median Error,2.3424
3,Std Dev,3.5011
4,Sample Count,4863.0000


In [71]:
calculate_aspect_summary(fire_growth_combined['aspect'], fire_growth_combined['aspect_v2'])

,Metric,Value
0,MAE,12.2309
1,RMSE,22.5760
2,Median Error,6.8223
3,Std Dev,18.9758
4,Sample Count,4863.0000


# Weather (ERA5)

# Requests

In [72]:
import xarray as xr
import cdsapi
import zipfile
from dateutil.relativedelta import relativedelta
import datetime
import dask

In [73]:
ERA5_FOLDER = 'ERA5'
os.makedirs(ERA5_FOLDER, exist_ok=True)

In [ ]:
# No need to run, the files are already provided

dataset = "reanalysis-era5-land"

year = 2024

for month in [3, 4, 5, 6, 7, 8]:

    request = {
        "variable": [
            "2m_dewpoint_temperature",
            "2m_temperature",
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "total_precipitation"
        ],
        "year": year,
        "month": month,
        "day": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15",
            "16", "17", "18",
            "19", "20", "21",
            "22", "23", "24",
            "25", "26", "27",
            "28", "29", "30", "31"
        ],
        "time": [
            "00:00", "01:00", "02:00",
            "03:00", "04:00", "05:00",
            "06:00", "07:00", "08:00",
            "09:00", "10:00", "11:00",
            "12:00", "13:00", "14:00",
            "15:00", "16:00", "17:00",
            "18:00", "19:00", "20:00",
            "21:00", "22:00", "23:00"
        ],
        # "data_format": "grib",
        "data_format": "netcdf",
        "download_format": "zip",
        "area": [84, -142, 41, -51]
    }
    
    path = os.path.join(ERA5_FOLDER, f"ERA5_LAND_{year}_{month}.zip")

    client = cdsapi.Client()
    client.retrieve(dataset, request, path)

    with zipfile.ZipFile(f"{ERA5_FOLDER}/ERA5_LAND_{year}_{month}.zip", "r") as z:
        z.extractall(f"{ERA5_FOLDER}/ERA5_LAND_{year}_{month}")

# Pipeline (2024 data)

In [74]:
def run_bulk_era5_pipeline_vectorized(df, ds_indexed, lon_col, lat_col):
    """
    Get the weather values from ERA5 files
    
    :param df: Dataframe
    :param ds_indexed: The ERA5 xarray Dataset
    :param lon_col: Longitude column in df
    :param lat_col: Latitude column in df
    """

    # Target points
    lats = xr.DataArray(df[lat_col], dims="point")
    lons = xr.DataArray(df[lon_col], dims="point")

    # End time for the temperature (end of the day in UTC)
    t_end_times = xr.DataArray(pd.to_datetime(df['end_utc']), dims="point")

    # End time for the precipitation (noon time in UTC)
    p_end_times = xr.DataArray(pd.to_datetime(df['noon_utc']) + pd.Timedelta(hours=1), dims="point")
    
    # Noon time (for wind and RH)
    noon_times = xr.DataArray(pd.to_datetime(df['noon_utc']) + pd.Timedelta(hours=1), dims="point")

    # Fetch Snapshots (Wind and RH)
    noon_ds = ds_indexed.sel(latitude=lats, longitude=lons, valid_time=noon_times, method='nearest')
    u = noon_ds['u10'].values
    v = noon_ds['v10'].values
    df['ws_v2'] = np.sqrt(u**2 + v**2) * 3.6
    
    t_c_noon = noon_ds['t2m'].values - 273.15
    d_c_noon = noon_ds['d2m'].values - 273.15
    df['rh_v2'] = 100 * (((112 - 0.1 * t_c_noon + d_c_noon) / (112 + 0.9 * t_c_noon))**8)

    # Fetch Temperature and Precipitation at their respective end times
    tmax_ds = ds_indexed['t2m_24h_max'].sel(latitude=lats, longitude=lons, valid_time=t_end_times, method='nearest')
    df['tmax_v2'] = tmax_ds.values - 273.15

    prec_ds = ds_indexed['tp_24h_sum'].sel(latitude=lats, longitude=lons, valid_time=p_end_times, method='nearest')
    df['prec_v2'] = prec_ds.values * 1000  # Convert m to mm

    return df

In [75]:
def fast_deduplicate_by_data(ds):
    """
    Remove duplicates in the data when merging multiple months from ERA5
    Example: Month 5 and Month 6 both contain data for the day 31-05 
    However, in Month 6, the data only contains NaNs
    So we remove based on the day that contains NaN (that is what call later the qulity score)
    
    :param ds: The ERA5 xarray Dataset
    """

    
    # Create a 'quality score' (number of non-NaN values per timestamp)
    # We sum across space to see which hour is 'full'
    quality = (~ds['t2m'].isnull()).sum(dim=['latitude', 'longitude'])

    # Add this quality as a temporary coordinate
    ds = ds.assign_coords(quality=quality)
    
    # Sort by valid_time AND quality
    # This puts the timestamps with NaNs at the 'bottom' of each duplicate group
    ds = ds.sortby(['valid_time', 'quality'])
    
    # Drop duplicates, keeping the 'last' (which is the one with highest quality)
    ds = ds.drop_duplicates("valid_time", keep='last')
    
    return ds.drop_vars('quality')

In [77]:
%%time

FIRE_SUBSETS_FOLDER = 'Fire_Subsets' 

for csv_file in Path(FIRE_SUBSETS_FOLDER).glob("*.csv"):

    
    print(f"\n{'='*60}\nProcessing {csv_file.name}...")
    df = pd.read_csv(csv_file)
    
    unique_dates = [datetime.datetime(int(y), int(m), 1) for y, m in df[['year', 'month']].drop_duplicates().values]
    start_date = min(unique_dates) - relativedelta(months=1)
    end_date = max(unique_dates) + relativedelta(months=1) 

    # Fetching all the required months (months where the wildfire occurred)
    required_gribs = []
    current = start_date
    while current <= end_date:
        grib_path = Path(ERA5_FOLDER) / f"ERA5_LAND_{current.year}_{current.month}" / "data.grib"
        if grib_path.exists():
            required_gribs.append(grib_path)
        current += relativedelta(months=1)

    if not required_gribs:
        continue

    # Open with 'minimal' coords to prevent index conflicts
    ds = xr.open_mfdataset(required_gribs, engine="cfgrib", combine='by_coords', 
                           coords="minimal", compat="override")
    
    # Flatten and swap
    ds_stacked = ds.stack(forecast_time=("time", "step"))
    ds_indexed = ds_stacked.swap_dims({"forecast_time": "valid_time"}).sortby("valid_time")

    # Drop duplicates to avoid InvalidIndexError
    print('Starting deduplication ...')
    initial_len = len(ds_indexed.valid_time)
    ds_indexed = fast_deduplicate_by_data(ds_indexed)
    final_len = len(ds_indexed.valid_time)
    print(f"Removed {initial_len - final_len} duplicate timestamps.")

    print("Calculating Hourly Precipitation...")
    # The precipitation in ERA5 is cumulative, we need to subtract two successive steps to get the raw value
    # Additionnally, this accumulations resets everytime at midnight, so the precipitation at 01:00 must be kept the same
    tp_prev = ds_indexed['tp'].shift(valid_time=1)
    ds_indexed['tp_hourly'] = xr.where(
            (ds_indexed.valid_time.dt.hour == 1) | ((ds_indexed['tp'] - tp_prev) < -1e-7),
            ds_indexed['tp'],
            ds_indexed['tp'] - tp_prev
    )

    print("Calculating 24h Rolling Aggregates...")
    ds_indexed['t2m_24h_max'] = ds_indexed['t2m'].rolling(valid_time=24).max() # Max temperature over 24H
    ds_indexed['tp_24h_sum'] = ds_indexed['tp_hourly'].rolling(valid_time=24).sum() # Sum precipitation over 24H

    print(f"Applying pipeline to {len(df)} points...")
    processed_df = run_bulk_era5_pipeline_vectorized(df, ds_indexed, 'lon', 'lat')
    
    processed_df.to_csv(csv_file, index=False)
    print(f"Saved results to {csv_file}")
    
    ds.close()
    del ds, ds_stacked, ds_indexed, processed_df


Processing subset_fire_2024_188.csv...
Starting deduplication ...
Removed 48 duplicate timestamps.
Calculating Hourly Precipitation...
Calculating 24h Rolling Aggregates...
Applying pipeline to 4863 points...
Saved results to Fire_Subsets\subset_fire_2024_188.csv
CPU times: total: 10min 36s
Wall time: 8min 35s


In [79]:
for filename in os.listdir(FIRE_SUBSETS_FOLDER):
    
    # Construct the full path
    file_path = os.path.join(FIRE_SUBSETS_FOLDER, filename)
    
    if os.path.isfile(file_path):
        
        print(f"Processing: {filename}")
        subset_fire_df = pd.read_csv(file_path)
        print(subset_fire_df.shape)

Processing: subset_fire_2024_188.csv
(4863, 32)


In [80]:
# Quick verification
subset_fire_df = pd.read_csv(f'{FIRE_SUBSETS_FOLDER}/subset_fire_2024_188.csv')
subset_fire_df[subset_fire_df['fireday']==6][['DOB', 'lon', 'lat', 'tmax', 'tmax_v2', 'prec', 'prec_v2', 'rh', 'rh_v2', 'ws', 'ws_v2']]

,DOB,lon,lat,tmax,tmax_v2,prec,prec_v2,rh,rh_v2,ws,ws_v2
19,203,-125.611063,58.213007,23.156885,23.156890,0.3371,0.337067,49.775600,49.775547,10.368354,10.368354
35,203,-125.611728,58.211889,23.156885,23.156890,0.3371,0.337067,49.775600,49.775547,10.368354,10.368354
36,203,-125.610335,58.212272,23.156885,23.156890,0.3371,0.337067,49.775600,49.775547,10.368354,10.368354
37,203,-125.608943,58.212656,23.156885,23.156890,0.3371,0.337067,49.775600,49.775547,10.368354,10.368354
56,203,-125.612393,58.210770,23.156885,23.156890,0.3371,0.337067,49.775600,49.775547,10.368354,10.368354
...,...,...,...,...,...,...,...,...,...,...,...
3891,203,-125.453725,58.166529,22.258386,22.258453,0.3413,0.341329,50.765099,50.765070,10.223996,10.223996
3893,203,-125.450940,58.167293,22.258386,22.258453,0.3413,0.341329,50.765099,50.765070,10.223996,10.223996
3894,203,-125.449548,58.167675,21.365900,21.365875,0.3724,0.372368,52.368000,52.367980,10.110347,10.110347
3920,203,-125.448825,58.166939,21.365900,21.365875,0.3724,0.372368,52.368000,52.367980,10.110347,10.110347


In [85]:
def calculate_weather_relative_error(df, column_pairs, threshold=1e-6):

    error_df = df[['ID', 'DOB', 'lat', 'lon']].copy()
    
    for actual_col, pred_col in column_pairs:
        
        error_name = f"rel_error_{pred_col}"
        val_actual_name = f"val_{actual_col}"
        val_pred_name = f"val_{pred_col}"
        
        valid_mask = df[actual_col].notna() & df[pred_col].notna()
        
        # Extract data
        act = df.loc[valid_mask, actual_col]
        pre = df.loc[valid_mask, pred_col]
        
        # Rounding logic for precipitation
        if "prec" in actual_col.lower() or "prec" in pred_col.lower():
            act = act.round(4)
            pre = pre.round(4)
        
        # Zero check
        is_act_zero = act.abs() < threshold
        is_pre_zero = pre.abs() < threshold

        # Difference and Relative Error
        diff = (pre - act).abs()
        res = np.where(
            is_act_zero & is_pre_zero, 0.0,
            np.where(
                is_act_zero, np.nan,
                (diff / act.abs()) * 100
            )
        )
        
        # Fill the DataFrame
        # error_df.loc[valid_mask, val_actual_name] = act
        # error_df.loc[valid_mask, val_pred_name] = pre
        error_df.loc[valid_mask, error_name] = res
        
    return error_df

In [86]:
all_dfs = []

for filename in os.listdir(FIRE_SUBSETS_FOLDER):
    
    file_path = os.path.join(FIRE_SUBSETS_FOLDER, filename)
    
    # Check if it's a CSV file
    if os.path.isfile(file_path) and filename.endswith('.csv'):
        print(f"Reading: {filename}")
        
        # Read the file
        subset_df = pd.read_csv(file_path)
        
        # Optional: Add a column to track which file the data came from
        subset_df['source_file'] = filename
        
        all_dfs.append(subset_df)

fire_growth_combined = pd.concat(all_dfs, axis=0, ignore_index=True)

print(f"Total rows: {len(fire_growth_combined)}")
fire_growth_combined.head()

Reading: subset_fire_2024_188.csv
Total rows: 4863


,ID,DOB,year,fireday,firearea,prec,tmax,ws,rh,Biomass,...,end_utc,noon_utc,slope_v2,aspect_v2,dem_v2,ws_v2,rh_v2,tmax_v2,prec_v2,source_file
0,2024_188,230,2024,33,79.38,0.0119,16.844110,5.604006,42.307301,6.000000,...,2024-08-18T07:00:00,2024-08-17T19:00:00,30.003590,220.317291,1766.930542,5.604006,42.307316,16.844147,0.011936,subset_fire_2024_188.csv
1,2024_188,230,2024,33,79.38,0.0119,16.844110,5.604006,42.307301,6.000000,...,2024-08-18T07:00:00,2024-08-17T19:00:00,30.987848,217.293915,1691.551758,5.604006,42.307316,16.844147,0.011936,subset_fire_2024_188.csv
2,2024_188,230,2024,33,79.38,0.0119,16.844110,5.604006,42.307301,4.666667,...,2024-08-18T07:00:00,2024-08-17T19:00:00,29.787878,213.921417,1781.523682,5.604006,42.307316,16.844147,0.011936,subset_fire_2024_188.csv
3,2024_188,230,2024,33,79.38,0.0119,16.844110,5.604006,42.307301,4.666667,...,2024-08-18T07:00:00,2024-08-17T19:00:00,23.522255,191.596771,1770.898926,5.604006,42.307316,16.844147,0.011936,subset_fire_2024_188.csv
4,2024_188,230,2024,33,79.38,0.0059,17.597101,5.519924,40.667500,6.000000,...,2024-08-18T07:00:00,2024-08-17T19:00:00,23.923401,185.389313,1793.195923,5.519923,40.667473,17.597076,0.005826,subset_fire_2024_188.csv


In [87]:
pairs = [('tmax', 'tmax_v2'), ('prec', 'prec_v2'), ('rh', 'rh_v2'), ('ws', 'ws_v2')]
results = calculate_weather_relative_error(fire_growth_combined, pairs)

In [88]:
summary_stats = results.describe(percentiles=[0.5])
summary_stats_rounded = summary_stats.round(6)
summary_stats_rounded

,DOB,lat,lon,rel_error_tmax_v2,rel_error_prec_v2,rel_error_rh_v2,rel_error_ws_v2
count,4863.000000,4863.000000,4863.000000,4863.000000,4863.000000,4863.000000,4863.000000
mean,207.392350,58.183520,-125.549697,0.000139,0.003001,0.000163,0.000003
std,9.267289,0.012526,0.058616,0.000116,0.039784,0.000353,0.000003
min,198.000000,58.148558,-125.651308,0.000023,0.000000,0.000002,0.000000
50%,203.000000,58.183535,-125.560392,0.000166,0.000000,0.000056,0.000002
max,232.000000,58.214093,-125.435873,0.000608,1.694915,0.001470,0.000015


# Disturbance product (SCANFI Enhancement)

In [89]:
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import from_bounds
import rioxarray
from rasterio.windows import Window
from rasterio.windows import from_bounds
from pyproj import Transformer
from scipy import stats
import xarray as xr

In [ ]:
DISTURBANCE_FOLDER = 'Fuel_Disturbance'
os.makedirs(DISTURBANCE_FOLDER, exist_ok=True)

In [ ]:
%%time
fire_growth_2020 = pd.read_csv('Fire growth points/Firegrowth_pts_v1_1_2020/Firegrowth_pts_v1_1_2020.csv')

In [ ]:
%%time

target_folder = Path(DISTURBANCE_FOLDER)

tif_files = [f for f in target_folder.glob("*.tif") if "_90m" not in f.name]

for tif_path in tqdm(tif_files):
    # Generate the new filename
    output_path = tif_path.with_name(f"{tif_path.stem}_90m.tif")
    
    print(f"Processing: {tif_path.name} -> {output_path.name}")
    
    da = rioxarray.open_rasterio(tif_path)
    
    # Take the max of each 3x3 bloc (we are dealing with a categorical data)
    da_90m = da.coarsen(x=3, y=3, boundary="trim").max()
    
    # Save
    da_90m.rio.to_raster(output_path, tiled=True, compress="lzw")
    
    # Cleanup memory
    da.close()
    del da, da_90m

print("Batch processing complete!")

In [ ]:
# Test with the original raster (30m resolution)

target_folder = Path(DISTURBANCE_FOLDER)
raster_path = target_folder/'canlad_annual_2020_v1.tif'

with rioxarray.open_rasterio(raster_path) as da:
            
    transformer = Transformer.from_crs("EPSG:4269", da.rio.crs, always_xy=True)
    target_x, target_y = transformer.transform(fire_growth_2020['lon'].values, fire_growth_2020['lat'].values)

    print('Done')
            
    x_coords = xr.DataArray(target_x, dims="points")
    y_coords = xr.DataArray(target_y, dims="points")

    print('Done')
            
    sampled = da.sel(x=x_coords, y=y_coords, method="nearest").compute()

    print('Done')

    fire_growth_2020[f"disturbance"] = sampled.values[0]

In [ ]:
%%time

# Test with the created raster (90m resolution)

target_folder = Path(DISTURBANCE_FOLDER)
raster_path = target_folder/'canlad_annual_2020_v1_90m.tif'

with rioxarray.open_rasterio(raster_path) as da:
            
    transformer = Transformer.from_crs("EPSG:4269", da.rio.crs, always_xy=True)
    target_x, target_y = transformer.transform(fire_growth_2020['lon'].values, fire_growth_2020['lat'].values)

    print('Done')
            
    x_coords = xr.DataArray(target_x, dims="points")
    y_coords = xr.DataArray(target_y, dims="points")

    print('Done')
            
    sampled = da.sel(x=x_coords, y=y_coords, method="nearest").compute()

    print('Done')

    fire_growth_2020[f"disturbance_90m"] = sampled.values[0]

In [ ]:
fire_growth_2020[fire_growth_2020['disturbance']!=0][['ID', 'year', 'disturbance', 'disturbance_90m']]

In [ ]:
values, counts = np.unique(fire_growth_2020['disturbance'].values, return_counts=True)
dist_summary = dict(zip(values, counts))
dist_summary

In [ ]:
values, counts = np.unique(fire_growth_2020['disturbance_90m'].values, return_counts=True)
dist_summary = dict(zip(values, counts))
dist_summary

In [ ]:
np.unique(fire_growth_2020['disturbance'].values)

# Fuel (SCANFI)

In [90]:
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import from_bounds
import rioxarray
from rasterio.windows import Window
from rasterio.windows import from_bounds
from pyproj import Transformer
from scipy import stats
import xarray as xr
from dask.diagnostics import ProgressBar
import subprocess

In [91]:
SCANFI_FOLDER = 'SCANFI'
os.makedirs(SCANFI_FOLDER, exist_ok=True)

SCANFI_VARS = {
    'Biomass': 'SCANFI_att_biomass_SW',
    'Closure': 'SCANFI_att_closure_SW',
    'prcC': 'SCANFI_att_prcC_SW',
    'prcB': 'SCANFI_att_prcB_SW'
}
SCANFI_KEYWORDS = ['biomass', 'closure', 'prcC', 'prcB']

In [92]:
def process_scanfi_gdal(scanfi_folder_path, scale_factor=3):
    """
    Coarsens SCANFI .tif files by a given scale factor using GDAL average resampling.
    Safely ignores NoData values and prevents RAM crashes.
    """
    
    # Dynamic Path Setup for OSGeo4W (Windows)
    gdal_bin = Path(os.environ.get("USERPROFILE", "")) / "AppData/Local/Programs/OSGeo4W/bin"
    if not gdal_bin.exists():
        gdal_bin = Path("C:/OSGeo4W/bin")

    env = os.environ.copy()
    env["PATH"] = f"{gdal_bin}{os.pathsep}{env['PATH']}"
    
    gdalwarp = str(gdal_bin / "gdalwarp.exe")

    # Define target folder
    target_folder = Path(scanfi_folder_path) / "2020"
    
    if not target_folder.exists():
        print(f"Directory not found: {target_folder}")
        return

    # Grab all .tif files that haven't been processed yet
    tif_files = [f for f in target_folder.glob("*.tif") if "_90m" not in f.name]
    
    if not tif_files:
        print("No valid .tif files found to process.")
        return

    print(f"Found {len(tif_files)} files. Starting GDAL batch processing...\n")

    # Loop through files
    for tif_path in tqdm(tif_files, desc="Coarsening to 90m"):
        
        output_path = tif_path.with_name(f"{tif_path.stem}_90m_v2.tif")
            
        tqdm.write(f"\nProcessing: {tif_path.name}")
        
        with rasterio.open(tif_path) as src:
            # Get Resolution
            orig_res_x, orig_res_y = src.res 
            target_res_x = orig_res_x * scale_factor
            target_res_y = orig_res_y * scale_factor
            # print(target_res_x, target_res_y)
            
            # Get NoData dynamically
            input_nodata = src.nodata 
            # print(input_nodata)

        # Build the Command List
        warp_cmd = [
            "gdalwarp", 
            "-ot", "Float32",
            "-tr", str(target_res_x), str(target_res_y),
            "-r", "average",
            "-wm", "4000",
            "-multi",
            "-co", "COMPRESS=ZSTD",
            "-co", "PREDICTOR=3",
            "-co", "TILED=YES",
            "-co", "NUM_THREADS=ALL_CPUS",
            "-overwrite"
        ]
        
        # Handle NoData mapping
        if input_nodata is not None:
            warp_cmd.extend(["-srcnodata", str(input_nodata), "-dstnodata", "nan"])
        
        # Add paths
        warp_cmd.extend([str(tif_path), str(output_path)])
        
        # Execution
        try:
            
            result = subprocess.run(warp_cmd, check=True, capture_output=True, text=True)
            print(f"Success: {os.path.basename(tif_path)} -> Float32 with Average Resampling")
        
        except subprocess.CalledProcessError as e:
            
            print(f"Error processing {tif_path}")
            print(f"GDAL Message: {e.stderr}")
        
    print("\nBatch processing complete!")

In [ ]:
process_scanfi_gdal(SCANFI_FOLDER)

In [93]:
import rasterio

with rasterio.open('SCANFI/2020/SCANFI_sps_prcC_other_SW_2020_90m_v2.tif') as src:
    print(f"Data Type: {src.dtypes[0]}") # e.g., 'float32', 'int16'
    print(f"NoData Value: {src.nodata}")
    print(f"Shape: {src.shape}")

Data Type: float32
NoData Value: nan
Shape: (39700, 59467)


### SCANFI 2020 data

In [94]:
%%time
fire_growth_2020 = pd.read_csv('Fire growth points/Firegrowth_pts_v1_1_2020/Firegrowth_pts_v1_1_2020.csv')

CPU times: total: 5.33 s
Wall time: 5.54 s


In [95]:
%%time 

current_year = '2020'
year_folder = Path(SCANFI_FOLDER) / str(current_year)

for keyword in tqdm(SCANFI_KEYWORDS):

    # matching_files = list(year_folder.glob(f"*{keyword}*_90m.tif"))
    matching_files = list(year_folder.glob(f"*{keyword}*_90m_v2.tif"))
        
    if not matching_files:
        print(f"No file found containing '{keyword}' in {year_folder}")
        continue
    
    raster_path = matching_files[0]
        
    if not raster_path.exists():
        print(f"Warning: {raster_path.name} not found. Skipping {var_name}.")
        continue
    else:
        print(f"Processing file: {raster_path.name}.")
            

    with rioxarray.open_rasterio(raster_path) as da:

            # Project coordinates to the SCANFI raster's coordinates system
            transformer = Transformer.from_crs("EPSG:4269", da.rio.crs, always_xy=True)
            target_x, target_y = transformer.transform(fire_growth_2020['lon'].values, fire_growth_2020['lat'].values)

            print('Done')
            
            x_coords = xr.DataArray(target_x, dims="points")
            y_coords = xr.DataArray(target_y, dims="points")

            print('Done')
            
            sampled = da.sel(x=x_coords, y=y_coords, method="nearest").compute()

            print('Done')

            fire_growth_2020[f"{keyword.lower()}_v2"] = sampled.values[0]

  0%|                                                                                            | 0/4 [00:00<?, ?it/s]

Processing file: SCANFI_att_biomass_SW_2020_90m_v2.tif.
Done
Done
Done


 25%|█████████████████████                                                               | 1/4 [00:27<01:23, 27.77s/it]

Processing file: SCANFI_att_closure_SW_2020_90m_v2.tif.
Done
Done
Done


 50%|██████████████████████████████████████████                                          | 2/4 [00:51<00:51, 25.62s/it]

Processing file: SCANFI_sps_prcC_other_SW_2020_90m_v2.tif.
Done
Done
Done


 75%|███████████████████████████████████████████████████████████████                     | 3/4 [01:15<00:24, 24.55s/it]

Processing file: SCANFI_sps_prcB_SW_2020_90m_v2.tif.
Done
Done
Done


100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [01:37<00:00, 24.41s/it]

CPU times: total: 1min 30s
Wall time: 1min 37s


In [96]:
fire_growth_2020['prcC_v3'] = 100 - fire_growth_2020['prcc_v2']

In [97]:
fire_growth_2020[['ID', 'year', 'Biomass', 'biomass_v2', 'Closure', 'closure_v2', 'prcB', 'prcb_v2', 'prcC', 'prcC_v3', 'prcc_v2']].head()

,ID,year,Biomass,biomass_v2,Closure,closure_v2,prcB,prcb_v2,prcC,prcC_v3,prcc_v2
0,2020_210,2020,16.777779,14.777778,23.888889,25.333334,10.666667,6.666667,89.333333,93.888885,6.111111
1,2020_210,2020,19.666666,0.000000,17.666666,0.000000,0.888889,0.000000,99.111111,100.000000,0.000000
2,2020_210,2020,17.777779,8.777778,26.555555,17.222221,3.888889,12.444445,96.111111,97.555557,2.444444
3,2020_210,2020,24.444445,23.555555,30.111111,41.111111,8.333333,0.888889,91.666667,92.777779,7.222222
4,2020_210,2020,17.222221,3.000000,13.333333,7.000000,10.666667,20.000000,89.333333,96.666664,3.333333
